In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras import *
from tensorflow.keras.applications import VGG19 
from tensorflow.keras.applications.vgg19 import preprocess_input
from tensorflow.keras.utils import to_categorical 

(xtrain, ytrain), (xtest, ytest) = mnist.load_data()
xtrain = tf.image.resize(xtrain[...,None],(32,32))
xtest = tf.image.resize(xtest[...,None],(32,32))

xtrain = tf.image.grayscale_to_rgb(xtrain).numpy()
xtest = tf.image.grayscale_to_rgb(xtest).numpy()

xtrain = preprocess_input(xtrain)
xtest = preprocess_input(xtest)

ytrain = to_categorical(ytrain, 10)
ytest = to_categorical(ytest, 10)

def create_model():
    base = VGG19(weights="imagenet",include_top=False,input_shape=(32,32,3))
    base.trainable = False
    x = layers.Flatten()(base.output)
    x = layers.Dense(512, activation="relu")(x)
    output = layers.Dense(10, activation="softmax")(x)

    model = Model(base.input,output)

    return base, model

base,model=create_model()
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
h1=model.fit(xtrain,ytrain,epochs=5,batch_size=64,validation_split=0.1)
loss1,acc1=model.evaluate(xtest,ytest,verbose=0)

base,model_ft=create_model()
base.trainable=True

for layer in base.layers[:4]:
    layer.trainable=False

model_ft.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
h2=model_ft.fit(xtrain,ytrain,epochs=5,batch_size=64,validation_split=0.1)
loss2,acc2=model_ft.evaluate(xtest,ytest,verbose=0)

print("Fixed Accuracy : ",acc1*100)
print("Fine tuning accruacy : ",acc2*100)
print("Fixed Loss : ",loss1)
print("Fine tuning loss :",loss2)

fig,ax=plt.subplots(2,2,figsize=(12,8))
ax[0,0].plot(h1.history['accuracy'])
ax[0,0].plot(h1.history['val_accuracy'])
ax[0,0].set_title("Fixed feature extraction")
ax[0,0].set_xlabel("Epochs")
ax[0,0].set_ylabel("accuracy")
ax[0,0].legend(["Train","Validation"])

ax[0,1].plot(h1.history['accuracy'])
ax[0,1].plot(h1.history['val_accuracy'])
ax[0,1].set_title("Fixed feature extraction")
ax[0,1].set_xlabel("Epochs")
ax[0,1].set_ylabel("accuracy")
ax[0,1].legend(["Train","Validation"])

ax[1,0].plot(h2.history['accuracy'])
ax[1,0].plot(h2.history['val_accuracy'])
ax[1,0].set_title("Fixed feature extraction")
ax[1,0].set_xlabel("Epochs")
ax[1,0].set_ylabel("accuracy")
ax[1,0].legend(["Train","Validation"])

ax[1,1].plot(h2.history['accuracy'])
ax[1,1].plot(h2.history['val_accuracy'])
ax[1,1].set_title("Fixed feature extraction")
ax[1,1].set_xlabel("Epochs")
ax[1,1].set_ylabel("accuracy")
ax[1,1].legend(["Train","Validation"])

plt.tight_layout()
plt.show()

Epoch 1/5
 32/844 ━━━━━━━━━━━━━━━━━━━━ 25:04 2s/step - accuracy: 0.3837 - loss: 6.1207